In [1]:
from torch import nn
import torch
from torch.utils.data import Subset, Dataset, DataLoader, random_split
from torchvision import transforms, models
from PIL import Image
import os
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter

In [2]:
class FlickrImg(Dataset):
 
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.paths = [
            os.path.join(image_dir, f)
            for f in os.listdir(image_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]
        self.transform = transform
 
    def __len__(self):
        return len(self.paths)
 
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img
 
def build_transforms(image_size=128):
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(image_size, scale=(0.85, 1.0), ratio=(0.95, 1.05)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),  # -> [-1, 1]
    ])
 
    val_transform = transforms.Compose([
        transforms.Resize(image_size),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ])
 
    return train_transform, val_transform


In [3]:
dataset_path = 'data/Images'
train_transform, val_transform = build_transforms()
train_dataset = FlickrImg(dataset_path, transform=train_transform)
val_dataset = FlickrImg(dataset_path, transform=val_transform)

val_len = int(len(train_dataset) * 0.1)
train_len = len(train_dataset) - val_len

generator = torch.Generator().manual_seed(42)
train_indices, val_indices = random_split(range(len(train_dataset)), [train_len, val_len], generator=generator)

train_subset = Subset(train_dataset, train_indices)
val_subset = Subset(val_dataset, val_indices)

train_loader = DataLoader(train_subset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_subset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)

len(train_subset), len(val_subset)


(28605, 3178)

In [4]:
class ResBlock(nn.Module):
 
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.GroupNorm(8, in_ch),
            nn.SiLU(),
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_ch),
            nn.SiLU(),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
        )
        self.skip = nn.Conv2d(in_ch, out_ch, kernel_size=1) if in_ch != out_ch else nn.Identity()
 
    def forward(self, x):
        return self.block(x) + self.skip(x)


In [5]:
class SelfAttention2d(nn.Module):
 
    def __init__(self, channels):
        super().__init__()
        self.norm = nn.GroupNorm(8, channels)
        self.q = nn.Conv2d(channels, channels, kernel_size=1)
        self.k = nn.Conv2d(channels, channels, kernel_size=1)
        self.v = nn.Conv2d(channels, channels, kernel_size=1)
        self.proj_out = nn.Conv2d(channels, channels, kernel_size=1)
        self.scale = channels ** -0.5
 
    def forward(self, x):
        b, c, h, w = x.shape
        h_ = self.norm(x)
        q = self.q(h_).reshape(b, c, h * w).permute(0, 2, 1)  # (B, HW, C)
        k = self.k(h_).reshape(b, c, h * w)                   # (B, C, HW)
        v = self.v(h_).reshape(b, c, h * w).permute(0, 2, 1)  # (B, HW, C)
 
        attn = torch.softmax(torch.bmm(q, k) * self.scale, dim=-1)  # (B, HW, HW)
        out = torch.bmm(attn, v)                                    # (B, HW, C)
        out = out.permute(0, 2, 1).reshape(b, c, h, w)
        return x + self.proj_out(out)

class Downsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, kernel_size=3, stride=2, padding=1)
 
    def forward(self, x):
        return self.conv(x)
 
 
class Upsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, kernel_size=3, padding=1)
 
    def forward(self, x):
        x = F.interpolate(x, scale_factor=2, mode='nearest')
        return self.conv(x)



In [6]:
class VAEEncoder(nn.Module):
 
    def __init__(self, latent_channels=4, base_ch=64):
        super().__init__()
        self.in_conv = nn.Conv2d(3, base_ch, kernel_size=3, padding=1)
 
        self.stage1 = nn.Sequential(ResBlock(base_ch, base_ch), Downsample(base_ch))
        self.stage2 = nn.Sequential(ResBlock(base_ch, base_ch * 2), Downsample(base_ch * 2))
        self.stage3 = nn.Sequential(ResBlock(base_ch * 2, base_ch * 4), Downsample(base_ch * 4))
 
        self.mid_block1 = ResBlock(base_ch * 4, base_ch * 4)
        self.mid_attn = SelfAttention2d(base_ch * 4)
        self.mid_block2 = ResBlock(base_ch * 4, base_ch * 4)
 
        self.norm_out = nn.GroupNorm(8, base_ch * 4)
        self.conv_mu = nn.Conv2d(base_ch * 4, latent_channels, kernel_size=1)
        self.conv_logvar = nn.Conv2d(base_ch * 4, latent_channels, kernel_size=1)
 
    def forward(self, x):
        x = self.in_conv(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.mid_block1(x)
        x = self.mid_attn(x)
        x = self.mid_block2(x)
        x = F.silu(self.norm_out(x))
        return self.conv_mu(x), self.conv_logvar(x)


    
def reparameterize(mu, logvar):
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std

In [7]:
class VAEDecoder(nn.Module):
 
    def __init__(self, latent_channels=4, base_ch=64):
        super().__init__()
        self.in_conv = nn.Conv2d(latent_channels, base_ch * 4, kernel_size=3, padding=1)
 
        self.mid_block1 = ResBlock(base_ch * 4, base_ch * 4)
        self.mid_attn = SelfAttention2d(base_ch * 4)
        self.mid_block2 = ResBlock(base_ch * 4, base_ch * 4)
 
        self.stage1 = nn.Sequential(ResBlock(base_ch * 4, base_ch * 2), Upsample(base_ch * 2))
        self.stage2 = nn.Sequential(ResBlock(base_ch * 2, base_ch), Upsample(base_ch))
        self.stage3 = nn.Sequential(ResBlock(base_ch, base_ch), Upsample(base_ch))
 
        self.norm_out = nn.GroupNorm(8, base_ch)
        self.out_conv = nn.Conv2d(base_ch, 3, kernel_size=3, padding=1)
 
    def forward(self, z):
        x = self.in_conv(z)
        x = self.mid_block1(x)
        x = self.mid_attn(x)
        x = self.mid_block2(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = F.silu(self.norm_out(x))
        return torch.tanh(self.out_conv(x))


In [8]:
class VGGPerceptualLoss(nn.Module):
 
    def __init__(self, layer_ids=(3, 8, 15, 22)):
        super().__init__()
        vgg = models.vgg16(weights='DEFAULT').features
        self.layer_ids = set(layer_ids)
        self.vgg = vgg
        for p in self.vgg.parameters():
            p.requires_grad = False
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
 
    def _normalize(self, x):
        x = (x + 1) / 2
        return (x - self.mean) / self.std
 
    def forward(self, recon, target):
        recon_n = self._normalize(recon)
        target_n = self._normalize(target)
 
        loss = 0.0
        x_r, x_t = recon_n, target_n
        for i, layer in enumerate(self.vgg):
            x_r = layer(x_r)
            with torch.no_grad():
                x_t = layer(x_t)
            if i in self.layer_ids:
                loss = loss + F.l1_loss(x_r, x_t)
            if i == max(self.layer_ids):
                break
        return loss


def vae_loss(recon, target, mu, logvar, perceptual_loss_fn, kl_weight=1e-6, perceptual_weight=0.5):
    recon_l1 = F.l1_loss(recon, target)
    perceptual = perceptual_loss_fn(recon, target)
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
 
    total = recon_l1 + perceptual_weight * perceptual + kl_weight * kl
    return total, {
        'recon_l1': recon_l1.item(),
        'perceptual': perceptual.item(),
        'kl': kl.item(),
        'total': total.item(),
    }


def kl_weight_schedule(step, warmup_steps, target_kl_weight):
    if step >= warmup_steps:
        return target_kl_weight
    return target_kl_weight * (step / warmup_steps)

    


In [9]:
@torch.no_grad()
def validate(encoder, decoder, perceptual_loss_fn, val_loader, device, kl_weight):
    encoder.eval()
    decoder.eval()
 
    total_logs = {'recon_l1': 0.0, 'perceptual': 0.0, 'kl': 0.0, 'total': 0.0}
    n_batches = 0
 
    for images in val_loader:
        images = images.to(device)
        mu, logvar = encoder(images)
        z = reparameterize(mu, logvar)
        recon = decoder(z)
        _, logs = vae_loss(recon, images, mu, logvar, perceptual_loss_fn, kl_weight=kl_weight)
 
        for k in total_logs:
            total_logs[k] += logs[k]
        n_batches += 1
 
    return {k: v / n_batches for k, v in total_logs.items()}


In [10]:
def train(
    output_dir='model_weights',
    num_epochs=100,
    lr=1e-4,
    target_kl_weight=1e-6,
    warmup_epochs=10,
    early_stopping_patience=12,
):
    os.makedirs(output_dir, exist_ok=True)
 
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    writer = SummaryWriter('RUNS')
 
    steps_per_epoch = len(train_loader)
    warmup_steps = warmup_epochs * steps_per_epoch

 
    encoder = VAEEncoder(latent_channels=4).to(device)
    decoder = VAEDecoder(latent_channels=4).to(device)
    perceptual_loss_fn = VGGPerceptualLoss().to(device)
 
    params = list(encoder.parameters()) + list(decoder.parameters())
    optimizer = torch.optim.AdamW(params, lr=lr)
 
    best_val_loss = float('inf')
    epochs_without_improvement = 0
    global_step = 0
 
    for epoch in tqdm(range(num_epochs)):
        epoch_logs = {'recon_l1': 0.0, 'perceptual': 0.0, 'kl': 0.0, 'total': 0.0}
        n_batches = 0
 
        for images in train_loader:
            images = images.to(device)
            kl_weight = kl_weight_schedule(global_step, warmup_steps, target_kl_weight)
 
            optimizer.zero_grad()
            mu, logvar = encoder(images)
            z = reparameterize(mu, logvar)
            recon = decoder(z)
            loss, logs = vae_loss(recon, images, mu, logvar, perceptual_loss_fn, kl_weight=kl_weight)
            loss.backward()
            optimizer.step()
 
            for k, v in logs.items():
                epoch_logs[k] += v
            n_batches += 1
            global_step += 1
        train_avg = {k: v / n_batches for k, v in epoch_logs.items()}
 
        print(f"epoch {epoch} "
              f"recon={train_avg['recon_l1']:.4f} perceptual={train_avg['perceptual']:.4f} "
              f"kl={train_avg['kl']:.2f} total={train_avg['total']:.4f}")
 
        for k, v in train_avg.items():
            writer.add_scalar(f'train/{k}', v, epoch)
        writer.add_scalar('train/kl_weight', kl_weight, epoch)
 
        val_logs = validate(encoder, decoder, perceptual_loss_fn, val_loader, device, target_kl_weight)
        print(f"[val] epoch {epoch} recon={val_logs['recon_l1']:.4f} "
              f"perceptual={val_logs['perceptual']:.4f} kl={val_logs['kl']:.2f} "
              f"total={val_logs['total']:.4f}")
 
        for k, v in val_logs.items():
            writer.add_scalar(f'val/{k}', v, epoch)
 
        if val_logs['total'] < best_val_loss:
            best_val_loss = val_logs['total']
            epochs_without_improvement = 0
            torch.save({
                'encoder': encoder.state_dict(),
                'decoder': decoder.state_dict(),
                'epoch': epoch,
                'val_loss': best_val_loss,
            }, os.path.join(output_dir, 'best_vae.pt'))
            print(f"  -> new best val loss {best_val_loss:.4f}, checkpoint saved")
        else:
            epochs_without_improvement += 1
            print(f"  -> no improvement ({epochs_without_improvement}/{early_stopping_patience})")
            if epochs_without_improvement >= early_stopping_patience:
                print(f"early stopping at epoch {epoch}")
                break
 
    writer.close()
    print("training finished, best val loss:", best_val_loss)
 


In [11]:
train()

  0%|          | 0/100 [00:09<?, ?it/s]


KeyboardInterrupt: 